### prepair modules and bases settings

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix,  classification_report, log_loss
from sklearn.preprocessing import StandardScaler
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
from scipy.stats import norm
import scipy.io
import re
import itertools

import os
from os.path import join
import contextlib
from copy import deepcopy
import imp 
import time 
import sys

import pickle
from pdb import set_trace

from IPython.display import clear_output, display

In [2]:
# Add the directory containing your modules to the Python path
sys.path.append(os.path.abspath(os.path.join('..', 'ses2_modelstims')))

# load local functions
import stim_io
import stim_io_plotting
import vtc
import bvbabel

/home/jorvhar/miniconda3/envs/predlis/lib/python3.8/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '


In [3]:
import regression
import varpar

In [4]:
## LOADING GRID

# Load from MAT file
variables = scipy.io.loadmat('/media/jorvhar/Data8T/MRIData/timing data/grid_parameters_python.mat')

# Extract individual variables
tunsteps = variables['tunsteps']
freqstep = variables['freqstep']
subsample = variables['subsample']
mustep = variables['mustep']
muarray_bins = variables['muarray_bins']
muarray = variables['muarray']
fwhm = variables['fwhm']
octgrid = variables['octgrid']
sigmagrid = variables['sigmagrid']
pref_range = variables['pref_range']
sharp_range_fwhm = variables['sharp_range_fwhm']
sharp_range = variables['sharp_range']

## 1. Set up regresiion model
Options:

In [8]:
### --- REGRESSION SAVING OPTIONS ---

# set modeltype
# modeltype = LinearRegression() #can be LinearRegression (ols), Ridge(alpha=..), Lasso(alpha=..)  etc.
modeltype = LinearRegression() 
key_ai = ['raw_scores', 'coefs', 'intercepts', 'correlation'] # what keys to median and mean across folds

# model return options
save_predict = False          # save y_pred-y
score_of_interest = 'score'   # what score to use  'score', 'raw_scores', 'coefs', 'intercepts', 'correlation'

# outlier options - #tobeimplemented
SD_lim = 3                    # remove y x sd higher then mean
remove_outliers = False       # if false dont remove sd outliers 

# what regressor variant to use
convolved = True   # use convolved dataset
resampled = True   # use scipy resampled data, instead of standard mean for downsampled data

zs=True  # zscore y
ts=True # temporally smooth y
hp=False  # highpass filter y

# add drift regressors
dr=False   # drift regressor

### --- LOCATION OPTIONS ---

# file location
mridat_dir = '/media/jorvhar/Data8T/MRIData/PreProc'
logdat_dir = '/media/jorvhar/Data8T/MRIData/timing data/data'
vtc_dir = '/media/jorvhar/New Volume1/vtcs' # adviced to put vtc's on a (nvme) ssd while running analyses
pp_dir = lambda pp, ses : f'S{pp:02d}_SES{ses}'
betas_dir = 'Betas'

# tonotopy and mask filenames
tonotopy_vmp = 'prf_permutations_for_s2.vmp'
mask_fn = 'gm-subcortical.msk'

# fn lambda
fn = lambda pp, ses, run : f'S{pp:02d}_SES{ses}_run{run}_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc'


### --- PARTICIPANT OPTIONS ---

# session of interest
ses = 2

# variable that may be different per participant
ppz = [1,2,3,4,5,6,7,8,9,10]
n_splitsz = [6,6,5,5,5,5,5,5,5,5]               # 12 runs > 10:2cross, 6 fold, splits used per pp (for variable length option)
n_splitsz = [12,12,10,10,10,10,10,10,10,10]                # 12 runs > 10:2cross, 6 fold, splits used per pp (for variable length option)
n_runz = [12,12,10,10,10,10,10,10,10,10]        # number of runs
startpp = 1

### --- SET THEORIE REGRESSION MODELS ---

# for full 3 sets we need 7 sets
# models = ['prediction',
#           'base_U_adaptation',
#           'base_U_adaptation_U_prediction']
models = ['base',
          'adaptation',
          'prediction',
          'base_U_adaptation',
          'base_U_prediction',
          'adaptation_U_prediction',
          'base_U_adaptation_U_prediction']
# if we want to use sets we only need 3 models 
##models=['base_U_adaptation', 'prediction', 'base_U_adaptation_U_prediction']

## REGRESSORS IN MODELS ##
model_regressors = {'base':       ['raw_acti', 'onoff'], 
                    'adaptation': ['raw_adapt' ],      # adaptation
                    'prediction': ['pred_prob',   # voxelwise prior liklihood
                                   'surprisal',   # global prior surprise
                                   'prec_w_surprisal',   # global prior surprise
                                   'precision']   # global precision
                                    
                   } 
# if we want to add adapted activation
# model_regressors['adaptation'] += ['adapt_activ']

# set combination of regressors
model_regressors.update({'base_U_adaptation':             model_regressors['base']+
                                                          model_regressors['adaptation'],
                        'base_U_prediction':              model_regressors['base']+
                                                          model_regressors['prediction'], 
                        'adaptation_U_prediction':        model_regressors['adaptation']+
                                                          model_regressors['prediction'], 
                        'base_U_adaptation_U_prediction': model_regressors['base']+
                                                          model_regressors['adaptation']+
                                                          model_regressors['prediction']})
y_var = 'voxeltimecourse'

## 2. Run regressions - per participant - per model - per gridpostion 
Run the full regressions, looping over participants, copy pasting files to a suitable ssd location, and doing the regression for the full grid.

In [9]:
# loop over all participants
for pp_idx in np.arange(ppz.index(startpp),len(ppz)):

    ### --- PREPARE PARTICIPANT DATA ---

    # fetch current pp vars
    pp = ppz[pp_idx]
    runz = np.arange(1,n_runz[pp_idx]+1)
    n_splits = n_splitsz[pp_idx]

    print(F'--RUNNING REGRESSION LOOP FOR PP: {pp} (runs={n_runz[pp_idx]},nr_splits={n_splits})--')

    # load stim df and tr df
    stim_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_stim_ideal')
    tr_df = stim_io.load_df(logdat_dir, pp, fn='processed_df_tr_ideal')

    # create full path for vmp and mask
    mskpath = join(mridat_dir, pp_dir(pp,1), mask_fn)
    vmppath = join(mridat_dir, pp_dir(pp,1), betas_dir, tonotopy_vmp)

    # load full mask and convert to indeces
    _, msk = bvbabel.msk.read_msk(mskpath)
    msk = np.where(msk)

    # load vmp image
    vmp_head, vmp_img = bvbabel.vmp.read_vmp(vmppath)

    # load list of filenames at origin, and vtc filenames
    origin_fns = [join(mridat_dir, pp_dir(pp, ses), fn(pp,ses,run)) for run in runz]
    vtc_fns = [join(vtc_dir, fn(pp,ses,run)) for run in runz]

    # copy files to ssd for efficient and fast chuck processing
    stim_io.copy_files(origin_fns, vtc_fns)

    # load tonotopy vmp
    vmp_df = stim_io.vmp_add_realsigma(vmp_img, msk, mustep[0][0]) # 1. prfMU, 2. prfMU_hz, prfS, prfO

    ### --- RUN FULL REGRESSION ---

    # run full regression for current pp
    scores = regression.run_model_grid(tr_df,stim_df,vmp_df,vtc_fns,
                                       msk, vmp_img,
                                       pref_range,sharp_range,
                                       models, model_regressors,
                                       mustep, n_splits, modeltype, key_ai,
                                       save_predict=save_predict, 
                                       convolved=convolved, resampled=resampled,
                                       zs=zs, ts=ts, hp=hp, dr=dr)
    # clean up prints - for next pp
    clear_output(wait=True)

    # save scores
    if not os.path.exists(join(mridat_dir, pp_dir(pp, ses), 'Betas')):
        os.mkdir(join(mridat_dir, pp_dir(pp, ses), 'Betas'))

    # append the pickle result naming based on cleaning steps 
    pick_fn = 'ANTS_IdealObserver_scores_pw_uw_lwo'
    if ts: pick_fn = f'{pick_fn}_tempsmooth'
    if hp: pick_fn = f'{pick_fn}_highpass'
    if dr: pick_fn = f'{pick_fn}_drift'
    # pickle the results
    with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'wb') as handle:
        pickle.dump(scores, handle, protocol=pickle.HIGHEST_PROTOCOL)
    # loading of pickled results
    ###with open(join(mridat_dir, pp_dir(pp, ses), f'Betas/{pick_fn}.pickle'), 'rb') as handle:
    ###    scores = pickle.load(handle)

    # clean up files where needed for next pp
    for fp in vtc_fns:
        os.remove(fp)



--RUNNING REGRESSION LOOP FOR PP: 10 (runs=10,nr_splits=10)--
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run1_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run2_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run3_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Volume1/vtcs/S10_SES2_run4_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc
Copied /media/jorvhar/Data8T/MRIData/PreProc/S10_SES2/S10_SES2_run5_FMR_SCSTBL_3DMAS_THPGLMF7c_TOPUP_ANTS.vtc to /media/jorvhar/New Vo

grid: 52/2400, 
        -current chuck took: 0.61 seconds
        -estimated time elapsed: 6.08 minutes of 280.58 minutes
grid: 53/2400, 
        -current chuck took: 1.27 seconds
        -estimated time elapsed: 6.10 minutes of 276.24 minutes
grid: 54/2400, 
        -current chuck took: 4.13 seconds
        -estimated time elapsed: 6.17 minutes of 274.19 minutes
grid: 55/2400, 
        -current chuck took: 4.01 seconds
        -estimated time elapsed: 6.24 minutes of 272.12 minutes
grid: 56/2400, 
        -current chuck took: 16.69 seconds
        -estimated time elapsed: 6.51 minutes of 279.19 minutes
grid: 57/2400, 
        -current chuck took: 1.97 seconds
        -estimated time elapsed: 6.55 minutes of 275.67 minutes
grid: 58/2400, 
        -current chuck took: 1.05 seconds
        -estimated time elapsed: 6.56 minutes of 271.64 minutes
grid: 59/2400, 
        -current chuck took: 15.63 seconds
        -estimated time elapsed: 6.83 minutes of 277.64 minutes
grid: 60/2400, 
      

grid: 119/2400, 
        -current chuck took: 10.27 seconds
        -estimated time elapsed: 11.35 minutes of 229.01 minutes
grid: 120/2400, 
        -current chuck took: 11.90 seconds
        -estimated time elapsed: 11.55 minutes of 231.07 minutes
grid: 121/2400, 
        -current chuck took: 1.48 seconds
        -estimated time elapsed: 11.58 minutes of 229.65 minutes
grid: 122/2400, 
        -current chuck took: 0.32 seconds
        -estimated time elapsed: 11.58 minutes of 227.87 minutes
grid: 123/2400, 
        -current chuck took: 1.19 seconds
        -estimated time elapsed: 11.60 minutes of 226.40 minutes
grid: 124/2400, 
        -current chuck took: 0.93 seconds
        -estimated time elapsed: 11.62 minutes of 224.88 minutes
grid: 125/2400, 
        -current chuck took: 6.48 seconds
        -estimated time elapsed: 11.73 minutes of 225.15 minutes
grid: 126/2400, 
        -current chuck took: 0.64 seconds
        -estimated time elapsed: 11.74 minutes of 223.57 minutes
grid: 

grid: 185/2400, 
        -current chuck took: 2.43 seconds
        -estimated time elapsed: 15.63 minutes of 202.82 minutes
grid: 186/2400, 
        -current chuck took: 2.57 seconds
        -estimated time elapsed: 15.68 minutes of 202.28 minutes
grid: 187/2400, 
        -current chuck took: 0.48 seconds
        -estimated time elapsed: 15.68 minutes of 201.30 minutes
grid: 188/2400, 
        -current chuck took: 0.40 seconds
        -estimated time elapsed: 15.69 minutes of 200.31 minutes
grid: 189/2400, 
        -current chuck took: 4.64 seconds
        -estimated time elapsed: 15.77 minutes of 200.24 minutes
grid: 190/2400, 
        -current chuck took: 1.72 seconds
        -estimated time elapsed: 15.80 minutes of 199.54 minutes
grid: 191/2400, 
        -current chuck took: 0.36 seconds
        -estimated time elapsed: 15.80 minutes of 198.58 minutes
grid: 192/2400, 
        -current chuck took: 1.25 seconds
        -estimated time elapsed: 15.82 minutes of 197.80 minutes
grid: 19

grid: 252/2400, 
        -current chuck took: 1.26 seconds
        -estimated time elapsed: 19.61 minutes of 186.74 minutes
grid: 253/2400, 
        -current chuck took: 0.70 seconds
        -estimated time elapsed: 19.62 minutes of 186.11 minutes
grid: 254/2400, 
        -current chuck took: 0.34 seconds
        -estimated time elapsed: 19.62 minutes of 185.43 minutes
grid: 255/2400, 
        -current chuck took: 1.09 seconds
        -estimated time elapsed: 19.64 minutes of 184.87 minutes
grid: 256/2400, 
        -current chuck took: 0.99 seconds
        -estimated time elapsed: 19.66 minutes of 184.31 minutes
grid: 257/2400, 
        -current chuck took: 0.84 seconds
        -estimated time elapsed: 19.67 minutes of 183.72 minutes
grid: 258/2400, 
        -current chuck took: 0.42 seconds
        -estimated time elapsed: 19.68 minutes of 183.08 minutes
grid: 259/2400, 
        -current chuck took: 8.41 seconds
        -estimated time elapsed: 19.82 minutes of 183.67 minutes
grid: 26

...

grid: 2098/2400, 
        -current chuck took: 6.33 seconds
        -estimated time elapsed: 170.71 minutes of 195.28 minutes
grid: 2099/2400, 
        -current chuck took: 1.74 seconds
        -estimated time elapsed: 170.74 minutes of 195.22 minutes
grid: 2100/2400, 
        -current chuck took: 6.60 seconds
        -estimated time elapsed: 170.85 minutes of 195.26 minutes
grid: 2101/2400, 
        -current chuck took: 1.66 seconds
        -estimated time elapsed: 170.88 minutes of 195.19 minutes
grid: 2102/2400, 
        -current chuck took: 0.96 seconds
        -estimated time elapsed: 170.89 minutes of 195.12 minutes
grid: 2103/2400, 
        -current chuck took: 1.63 seconds
        -estimated time elapsed: 170.92 minutes of 195.06 minutes
grid: 2104/2400, 
        -current chuck took: 1.36 seconds
        -estimated time elapsed: 170.94 minutes of 194.99 minutes
grid: 2105/2400, 
        -current chuck took: 8.74 seconds
        -estimated time elapsed: 171.09 minutes of 195.06 

grid: 2163/2400, 
        -current chuck took: 0.87 seconds
        -estimated time elapsed: 174.94 minutes of 194.11 minutes
grid: 2164/2400, 
        -current chuck took: 1.47 seconds
        -estimated time elapsed: 174.97 minutes of 194.05 minutes
grid: 2165/2400, 
        -current chuck took: 0.85 seconds
        -estimated time elapsed: 174.98 minutes of 193.98 minutes
grid: 2166/2400, 
        -current chuck took: 7.05 seconds
        -estimated time elapsed: 175.10 minutes of 194.02 minutes
grid: 2167/2400, 
        -current chuck took: 6.98 seconds
        -estimated time elapsed: 175.22 minutes of 194.06 minutes
grid: 2168/2400, 
        -current chuck took: 0.99 seconds
        -estimated time elapsed: 175.23 minutes of 193.98 minutes
grid: 2169/2400, 
        -current chuck took: 0.94 seconds
        -estimated time elapsed: 175.25 minutes of 193.91 minutes
grid: 2170/2400, 
        -current chuck took: 10.39 seconds
        -estimated time elapsed: 175.42 minutes of 194.01

grid: 2228/2400, 
        -current chuck took: 3.98 seconds
        -estimated time elapsed: 178.50 minutes of 192.28 minutes
grid: 2229/2400, 
        -current chuck took: 3.22 seconds
        -estimated time elapsed: 178.55 minutes of 192.25 minutes
grid: 2230/2400, 
        -current chuck took: 7.70 seconds
        -estimated time elapsed: 178.68 minutes of 192.30 minutes
grid: 2231/2400, 
        -current chuck took: 4.31 seconds
        -estimated time elapsed: 178.75 minutes of 192.29 minutes
grid: 2232/2400, 
        -current chuck took: 1.36 seconds
        -estimated time elapsed: 178.78 minutes of 192.23 minutes
grid: 2233/2400, 
        -current chuck took: 0.91 seconds
        -estimated time elapsed: 178.79 minutes of 192.16 minutes
grid: 2234/2400, 
        -current chuck took: 2.03 seconds
        -estimated time elapsed: 178.83 minutes of 192.11 minutes
grid: 2235/2400, 
        -current chuck took: 4.61 seconds
        -estimated time elapsed: 178.90 minutes of 192.11 

grid: 2293/2400, 
        -current chuck took: 1.37 seconds
        -estimated time elapsed: 182.90 minutes of 191.44 minutes
grid: 2294/2400, 
        -current chuck took: 0.88 seconds
        -estimated time elapsed: 182.92 minutes of 191.37 minutes
grid: 2295/2400, 
        -current chuck took: 0.91 seconds
        -estimated time elapsed: 182.93 minutes of 191.30 minutes
grid: 2296/2400, 
        -current chuck took: 8.79 seconds
        -estimated time elapsed: 183.08 minutes of 191.37 minutes
grid: 2297/2400, 
        -current chuck took: 8.32 seconds
        -estimated time elapsed: 183.22 minutes of 191.43 minutes
grid: 2298/2400, 
        -current chuck took: 1.50 seconds
        -estimated time elapsed: 183.24 minutes of 191.38 minutes
grid: 2299/2400, 
        -current chuck took: 1.70 seconds
        -estimated time elapsed: 183.27 minutes of 191.32 minutes
grid: 2300/2400, 
        -current chuck took: 9.27 seconds
        -estimated time elapsed: 183.43 minutes of 191.40 

grid: 2359/2400, 
        -current chuck took: 4.28 seconds
        -estimated time elapsed: 186.78 minutes of 190.03 minutes
grid: 2360/2400, 
        -current chuck took: 9.29 seconds
        -estimated time elapsed: 186.93 minutes of 190.10 minutes
grid: 2361/2400, 
        -current chuck took: 3.87 seconds
        -estimated time elapsed: 187.00 minutes of 190.09 minutes
grid: 2362/2400, 
        -current chuck took: 2.74 seconds
        -estimated time elapsed: 187.04 minutes of 190.05 minutes
grid: 2363/2400, 
        -current chuck took: 1.47 seconds
        -estimated time elapsed: 187.07 minutes of 190.00 minutes
grid: 2364/2400, 
        -current chuck took: 0.96 seconds
        -estimated time elapsed: 187.09 minutes of 189.93 minutes
grid: 2365/2400, 
        -current chuck took: 6.57 seconds
        -estimated time elapsed: 187.20 minutes of 189.97 minutes
grid: 2366/2400, 
        -current chuck took: 2.04 seconds
        -estimated time elapsed: 187.23 minutes of 189.92 